In [0]:
import requests

url = "https://www.unb.ca/cic/datasets/ids-2017.html"

try:
    response = requests.get(url, timeout=30)

    print("HTTP Status:", response.status_code)
    print("Final URL:", response.url)

    if response.status_code == 200:
        print("SUCCESS: CIC website is accessible!")
    else:
        print("Website responded, but returned a non-200 status.")

except Exception as e:
    print("CONNECTION FAILED:")
    print(type(e).__name__, str(e))

In [0]:
%sql
SELECT
    current_catalog() AS catalog,
    current_schema() AS schema;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.cic_ids2017;

In [0]:
DESCRIBE VOLUME workspace.default.cic_ids2017;

In [0]:
%sql
DESCRIBE VOLUME workspace.default.cic_ids2017;

In [0]:
import requests

url = "http://205.174.165.80/CICDataset/CIC-IDS-2017/Dataset/CIC-IDS-2017/CSVs/MachineLearningCSV.zip"

try:
    response = requests.get(
        url,
        headers={"Range": "bytes=0-0"},
        stream=True,
        timeout=30
    )

    print("HTTP Status:", response.status_code)
    print("Content Type:", response.headers.get("Content-Type"))
    print("Content Length:", response.headers.get("Content-Length"))
    print("Content Range:", response.headers.get("Content-Range"))

    response.close()

except Exception as e:
    print("ERROR:", str(e))

In [0]:
import requests

url = "http://205.174.165.80/CICDataset/CIC-IDS-2017/Dataset/CIC-IDS-2017/CSVs/MachineLearningCSV.zip"

response = requests.get(
    url,
    headers={"Range": "bytes=0-1023"},
    timeout=30
)

print("Final URL:", response.url)
print("Status:", response.status_code)
print("Content-Type:", response.headers.get("Content-Type"))
print("\nResponse preview:")
print(response.text[:500])

In [0]:
import requests

filename = "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"

url = (
    "https://huggingface.co/datasets/c01dsnap/"
    "CIC-IDS2017/resolve/main/" + filename
)

response = requests.get(
    url,
    headers={"Range": "bytes=0-0"},
    stream=True,
    timeout=60
)

print("HTTP Status:", response.status_code)
print("Content Type:", response.headers.get("Content-Type"))
print("Content Range:", response.headers.get("Content-Range"))
print("Final URL:", response.url[:150])

response.close()

In [0]:
import requests
from pathlib import Path

# Dataset source
filename = "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"

url = (
    "https://huggingface.co/datasets/c01dsnap/"
    "CIC-IDS2017/resolve/main/" + filename
)

# Unity Catalog Volume destination
folder = Path("/Volumes/workspace/default/cic_ids2017/raw")
folder.mkdir(parents=True, exist_ok=True)

output_path = folder / filename

print("Starting download:", filename)

downloaded = 0

try:
    with requests.get(url, stream=True, timeout=(30, 180)) as response:

        response.raise_for_status()

        print("HTTP Status:", response.status_code)
        print("Content Type:", response.headers.get("Content-Type"))

        expected_size = int(response.headers.get("Content-Length", 0))

        with open(output_path, "wb") as file:
            for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                if chunk:
                    file.write(chunk)
                    downloaded += len(chunk)

        if expected_size and downloaded != expected_size:
            raise ValueError(
                f"Incomplete download: {downloaded}/{expected_size} bytes"
            )

    print("\nDownload completed!")
    print("File size:", round(downloaded / (1024**2), 2), "MB")
    print("Saved to:", output_path)

except Exception as e:
    if output_path.exists():
        output_path.unlink()

    print("Download failed:", str(e))

In [0]:
import requests
from pathlib import Path
import os

# Destination in Unity Catalog Volume
folder = Path("/Volumes/workspace/default/cic_ids2017/raw")
folder.mkdir(parents=True, exist_ok=True)

# Remaining 7 files from CIC-IDS2017
files = [
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv"
]

base_url = (
    "https://huggingface.co/datasets/c01dsnap/"
    "CIC-IDS2017/resolve/main/"
)

session = requests.Session()
successful = []
failed = []

for i, filename in enumerate(files, start=1):

    output_path = folder / filename
    temp_path = folder / (filename + ".part")

    # Skip an existing completed file
    if output_path.exists() and output_path.stat().st_size > 1_000_000:
        print(f"[{i}/7] Already exists: {filename}")
        successful.append(filename)
        continue

    url = base_url + filename

    print(f"\n[{i}/7] Downloading: {filename}", flush=True)

    try:
        downloaded = 0

        with session.get(
            url,
            stream=True,
            timeout=(30, 300)
        ) as response:

            response.raise_for_status()

            content_type = response.headers.get("Content-Type", "")
            if "text/html" in content_type.lower():
                raise ValueError(
                    f"Expected CSV but received {content_type}"
                )

            with open(temp_path, "wb") as file:
                for chunk in response.iter_content(
                    chunk_size=8 * 1024 * 1024
                ):
                    if chunk:
                        file.write(chunk)
                        downloaded += len(chunk)

        if downloaded < 1_000_000:
            raise ValueError(
                f"Unexpectedly small file: {downloaded} bytes"
            )

        # Publish only after the download completes
        os.replace(temp_path, output_path)

        size_mb = downloaded / (1024 ** 2)

        print(
            f"Completed: {filename} | {size_mb:.2f} MB",
            flush=True
        )

        successful.append(filename)

    except Exception as e:
        if temp_path.exists():
            temp_path.unlink()

        print(f"FAILED: {filename} | {e}", flush=True)
        failed.append(filename)

session.close()

print("\n========== DOWNLOAD SUMMARY ==========")
print("Successful:", len(successful), "/ 7")
print("Failed:", len(failed), "/ 7")

if failed:
    print("\nFiles to retry:")
    for filename in failed:
        print("-", filename)

In [0]:
from pathlib import Path

BASE_PATH = "/Volumes/workspace/default/cic_ids2017"

folders = [
    "raw",
    "processed",
    "splits",
    "models",
    "results",
    "reports"
]

for folder in folders:
    dbutils.fs.mkdirs(f"{BASE_PATH}/{folder}")
    print(f"Created/verified: {folder}")

print("\nVolume structure:")

display(dbutils.fs.ls(BASE_PATH))